In [60]:
# IMPORTS
import torchvision.models as models
import pandas as pd
import torch
import torch.nn as nn # just get that all up in here. 
import os
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


from torch import optim

from torch.optim.lr_scheduler import StepLR
from imblearn.over_sampling import SMOTE
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset, DataLoader

# for toy model, use a SUBSET so we aren't tearing our hair out trying to run this. 
from torch.utils.data import Subset
# same thing that we did for the MLP
from sklearn.metrics import (roc_auc_score, confusion_matrix, 
                             classification_report, fbeta_score)
import numpy as np



In [61]:
# CHEXPERT for later
# refer to this later when you wnat to instead of fine tuning a network like we have been doing
# want to actually use the chexpert one (bc of course they published it with a python library)
# and then finetune THAT on my dataset. 

# # Install torchxrayvision (has CheXNet pretrained on NIH + CheXpert)
# !pip install torchxrayvision

# import torchxrayvision as xrv
# model = xrv.models.DenseNet(weights="densenet121-res224-all")  # Trained on 14 diseases
# # Replace classifier for binary pneumonia
# model.classifier = nn.Linear(1024, 1)

In [62]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

In [63]:
class ChestXRayDataset(Dataset):
  def __init__(self, records, image_dir, transform=None):
    """
    records    : a dataframe slice (post train test split)
    image_dir : path to folder containing all .dcm.png files
    transform : torchvision transforms pipeline
    """
    self.records = records.reset_index(drop=True)
    self.image_dir = image_dir
    self.transform = transform

  def __len__(self):
    return len(self.records)

  def __getitem__(self, idx):
    row = self.records.iloc[idx]
    img_path = os.path.join(self.image_dir, row['filename'])
    
    # Load as grayscale directly
    image = Image.open(img_path).convert('L')  # 'L' = grayscale
    
    if self.transform:
        image = self.transform(image)
    
    label = torch.tensor(row["label"], dtype=torch.float32)
    return image, label

In [64]:
import re

# Shared negation checker
def simple_negation_check(text, target_terms):
    """Return True if any target term is present and not negated."""
    if not isinstance(text, str):
        return False
    text_lower = text.lower()
    
    if not any(term in text_lower for term in target_terms):
        return False
    
    neg_patterns = [
        r'\bno\s+(?:\w+\s+){0,5}(?:' + '|'.join(target_terms) + r')\b',
        r'\bwithout\s+(?:\w+\s+){0,5}(?:' + '|'.join(target_terms) + r')\b',
        r'\bfree\s+of\s+(?:\w+\s+){0,5}(?:' + '|'.join(target_terms) + r')\b',
        r'\bnegative\s+for\s+(?:\w+\s+){0,5}(?:' + '|'.join(target_terms) + r')\b',
        r'\bnot\s+(?:\w+\s+){0,5}(?:' + '|'.join(target_terms) + r')\b',
    ]
    for pat in neg_patterns:
        if re.search(pat, text_lower):
            return False
    return True

PNEUMONIA_TERMS = ['pneumonia', 'consolidation', 'infiltrate', 'airspace disease', 'opacity']

def build_master_df(projections_path, reports_path, mode='pneumonia'):
    """
    Build master dataframe for CNN training.
    
    Args:
        projections_path: path to projections CSV
        reports_path: path to reports CSV
        mode: 'pneumonia' for pneumonia surrogate label
              'abnormal' for abnormal vs normal label
    
    Returns:
        DataFrame with columns ['filename', 'label']
    """
    print("This is the projections path ", projections_path)
    proj = pd.read_csv(projections_path)
    rep = pd.read_csv(reports_path)
    
    proj = proj[proj["projection"] == "Frontal"]
    
    if mode == 'pneumonia':
        # Create pneumonia label using free-text negation check
        rep["label"] = rep.apply(
            lambda row: 1 if simple_negation_check(
                f"{row.get('findings','')} {row.get('impression','')}",
                PNEUMONIA_TERMS
            ) else 0,
            axis=1
        )
        df = proj.merge(rep[['uid', 'label']], on='uid', how='inner')
        print(f"Pneumonia mode: {df['label'].sum()} positives / {len(df)} total ({df['label'].mean():.1%})")
        
    elif mode == 'abnormal':
        # Abnormal = anything not explicitly "normal"
        rep["label"] = rep["Problems"].apply(
            lambda x: 0 if str(x).lower().strip() == 'normal' else 1
        )
        df = proj.merge(rep[['uid', 'label']], on='uid', how='inner')
        print(f"Abnormal mode: {df['label'].sum()} positives / {len(df)} total ({df['label'].mean():.1%})")
        
    else:
        raise ValueError(f"Unknown mode '{mode}'. Use 'pneumonia' or 'abnormal'.")
    
    return df[["filename", "label"]]


# Backward compatibility aliases
def build_master_df_for_pneumonia(projections_path, reports_path):
    """Legacy wrapper for pneumonia mode."""
    return build_master_df(projections_path, reports_path, mode='pneumonia')

def build_master_df_for_abnormal(projections_path, reports_path):
    """Legacy wrapper for abnormal mode."""
    return build_master_df(projections_path, reports_path, mode='abnormal')

In [65]:
def get_dataloaders(projections_path, reports_path, image_dir, test_size=0.2,
                    batch_size=128, image_size=224, num_workers=0, mode="pneumonia", subset=False):
  # need to use 224 for 2 reasons: its what they used in the paper, and
  # its what ImageNet is trained on, so we need to ensure compatability. 

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  
  # this is wrong now. I need to fix this. 
  df = build_master_df(projections_path, reports_path, mode)

  train_df, test_df = train_test_split(
      df,
      test_size=test_size,
      random_state=42,
      stratify=df["label"] # make sure to get a good split. 
  )

  # DOING some standard imageNet transformations
  train_transforms = transforms.Compose([
      transforms.Resize((224, 224)), # start it a little bigger and then crop it down to 224 for some jitter. 
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),  # Works on grayscale too
        transforms.Grayscale(num_output_channels=1),  # Force 1 channel
        transforms.ToTensor(),
        # Single-channel normalization (mean=0.5, std=0.5 maps to [-1, 1])
        transforms.Normalize(mean=[0.5], std=[0.5])
  ])

  test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
  ])

  train_dataset = ChestXRayDataset(train_df, image_dir, transform=train_transforms)
  test_dataset = ChestXRayDataset(test_df, image_dir, transform=test_transforms)


  # go aheand and just do this here...
  n_neg = (train_df['label'] == 0).sum()
  n_pos = (train_df['label'] == 1).sum()
  pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)

  # if we wnat a subset, just create smaller ones, otherwise do nothing. 
  #  just rip this completely out. 
  subset = True # just do this here super quick. 

  if subset == True:
    train_dataset = Subset(train_dataset, range(200))
    test_dataset = Subset(test_dataset, range(50))


  train_loader = DataLoader(
      train_dataset,
      batch_size=batch_size,
      shuffle=True,
      num_workers=num_workers,
      pin_memory=True if device.type == 'cuda' else False  # speeds up GPU transfer... supposedly. 
  )

  test_loader = DataLoader(
      test_dataset,
      batch_size=batch_size,
      shuffle=False,   # ← never shuffle test
      num_workers=num_workers,
      pin_memory=True if device.type == 'cuda' else False
  )


  # get the df for NLP purposes, will probably find a better way to do it later.
  return train_loader, test_loader, pos_weight

In [66]:
class FinetunedDenseNet(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Load DenseNet-121 with pretrained ImageNet weights
        densenet = models.densenet121(weights="IMAGENET1K_V1")
        dropout = 0.5
        
        # Replace first conv with 1-channel version
        old_conv = densenet.features.conv0
        new_conv = nn.Conv2d(
            in_channels=1,  # Grayscale
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=old_conv.bias is not None
        )
        
        # Average RGB weights → Grayscale weights
        with torch.no_grad():
            new_conv.weight.data = old_conv.weight.data.mean(dim=1, keepdim=True)
            if old_conv.bias is not None:
                new_conv.bias.data = old_conv.bias.data
        
        densenet.features.conv0 = new_conv
        self.features = densenet.features
        num_features = 1024
    
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(num_features, 1)
        )
    def forward(self, x):
        x = self.features(x)      # x shape: (batch, 3, 224, 224) → (batch, 1024, 7, 7)
        x = self.classifier(x)    # → (batch, 1)
        return x

In [67]:
class ProgressiveUnfreezer:
    """ man I should make more docstrings like this, this is nuts. 
    Progressive unfreezing for Densenet121 
    usage:
        unfreezer = ProgresiveUnfreezer(model)

        for epoch in range(num_epochs):
            if freezer.should_update(epochs):
                optimizer = unfreezer.update(epoch, optmiizer)
    """

    def __init__(self, model, strategy="Progressive"):
        self.model = model
        self.strategy = strategy
        self.current_stage = 0
        self.stage_epochs = {0: 5, 1: 10, 2: 20} # could not tell you waht is happening here. 

        self._freeze_all()
        self._unfreeze_classifier()

    def _freeze_all(self):
        for param in self.model.parameters():
            param.requires_grad = False


    def _unfreeze_last_block(self):
        for name, p in self.model.features.named_parameters():
            if "denseblock4" in name or "norm5" in name:
                p.requires_grad = True

    def _unfreeze_classifier(self):
        for param in self.model.classifier.parameters():
            param.requires_grad = True
    

    # turns out this was faaar to aggressive. trying again. 
    # def _unfreeze_denseblock(self, block_num):
    #     # unfreeze a specific dense block (1-4)
    #     block_name = f'denseblock{block_num}'
    #     for name, param in self.model.features.named_parameters():
    #         if block_name in name:
    #             param.requires_grad = True

    # def _unfreeze_transition(self, trans_num):
    #     trans_name = f'transition{trans_num}'
    #     for name, param in self.model.features.named_parameters():
    #         if trans_name in name:
    #             param.requires_grad = True

    # ok how exactly does htis function work I no understand good 
    def get_stage_for_epoch(self, epoch):
        "Determin which stage of unfreezing we should be given the epoch"
        cumulative = 0
        for stage, ep in self.stage_epochs.items():
            cumulative += ep
            if epoch < cumulative:
                return stage
            
        return max(self.stage_epochs.keys())
    
    def should_update(self, epoch):
        # check if we need to change unfreezing stage
        new_stage = self.get_stage_for_epoch(epoch)
        return new_stage != self.current_stage
    
    def update(self, epoch, old_optimizer=None):
            new_stage = self.get_stage_for_epoch(epoch)
            if new_stage == self.current_stage:
                return old_optimizer
            
            print(f"\n🔓 Stage {new_stage}")
            if new_stage == 0:
                lr = 1e-3
                desc = "Classifier only"
            elif new_stage == 1:
                self._unfreeze_last_block()
                lr = 1e-4
                desc = "Classifier + DenseBlock4"
            elif new_stage == 2:
                # NEVER unfreeze everything—just add transition3
                for name, p in self.model.features.named_parameters():
                    if 'transition3' in name or 'denseblock3' in name:
                        p.requires_grad = True
                lr = 5e-5
                desc = "Classifier + DenseBlocks 3-4"
            
            self.current_stage = new_stage
            trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
            total = sum(p.numel() for p in self.model.parameters())
            print(f"  {desc} | LR: {lr} | Trainable: {trainable:,}/{total:,} ({100*trainable/total:.1f}%)")
            
            if old_optimizer:
                return torch.optim.AdamW(
                    filter(lambda p: p.requires_grad, self.model.parameters()),
                    lr=lr, weight_decay=1e-4
                )
            return None

        

In [68]:
# Constants
projections_path = '../IU-xray/indiana_projections.csv'
# this contains the labelling we want for our image CNN
reports_path = '../IU-xray/indiana_reports.csv'
# and of course, the images
image_dir = '../IU-xray/images/images_normalized'

In [69]:
def train_model(model, train_loader, val_loader, num_epochs=23, device="cuda", pos_weight=0.5, mode="penumonia"):

    model = model.to(device)

    # lets add some early stopping stuff

    best_val_auc = 0.0
    paitence_counter = 0
    paitence = 5
    best_epoch = 0
    bset_model_state = None

    unfreezer = ProgressiveUnfreezer(model)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), 
        lr=1e-3, 
        weight_decay=1e-4
    )

    # need to make sure this gets passed in. fantastic. 
    loss_function = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    torch.backends.cudnn.benchmark = True # apparently this is supposed to help
    

    best_val_auc = 0.0 # floor value
    history = {"train_loss": [], "val_loss": [], "val_auc": []}

    for epoch in range(num_epochs):
        # first lets make sure we have all the appropriate layers that we want. 
        if unfreezer.should_update(epoch):
            optimizer = unfreezer.update(epoch, optimizer)

        # training time
        model.train()
        train_loss = 0.0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True) # non blocking for even more speed
            labels = labels.to(device, non_blocking=True) # ditto


            optimizer.zero_grad(set_to_none=True) # apparently more efficient than zero_Grad()
            outputs = model(images) # take that whole batch in 
            loss = loss_function(outputs.squeeze(), labels) # ouptuts need to be squeezed to change shape from (B, 1) to just (B)
            # backward and optimize        
            # pros and cons of keeping the clip grad norm? working keeping?    
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            # get loss value and clean up
            train_loss += loss.item()

            del outputs, loss, images, labels # make sure to clean these up to keep the vram nice and neat :)

        
        # now for the validatino phase (we run these back to back for finetuning, less epochs)
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                outputs = model(images)
                loss = loss_function(outputs.squeeze(), labels)                
                val_loss += loss.item()

                preds = torch.sigmoid(outputs.squeeze())
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

                # lets get rid of this 
                # print(f"Val positives: {sum(all_labels)} / {len(all_labels)}")
                # print(f"Prediction mean: {np.mean(all_preds):.4f}, std: {np.std(all_preds):.4f}")   

        val_auc = roc_auc_score(all_labels, all_preds)

        # early stopping logic goes here. 
        if val_auc > best_val_auc + 0.001:
            best_val_auc = val_auc
            best_epoch = epoch
            patience_counter = 0 # reset
            bset_model_state = model.state_dict().copy()
            title = f"Best_densenet_{mode}.pth"
            torch.save(model.state_dict(), title)
            print(f"Best new model! AUC {best_val_auc:.4f}")
        else:
            paitence_counter += 1
            print("Paitence did not improve")

        
        if paitence_counter >= paitence:
            print("Early stoopping at epoch ", epoch)
            model.load_state_dict(bset_model_state)
            break


        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        print(f"Epoch {epoch+1}/{num_epochs} | "
                f"Train Loss: {avg_train_loss:.4f} | "
                f"Val Loss: {avg_val_loss:.4f} | "
                f"Val AUC: {val_auc:.4f}")
        
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), "Best_densenet_pneumonia.pth")
            print(f"New best model! AUC {best_val_auc:.4f}")

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        history["val_auc"].append(val_auc)

    return history




In [ ]:
# aight lets just see if this works

model = FinetunedDenseNet()
model = model.to(device)
# no longer need to do the weird he initialization, as I am using a pre-trained model. 

# lets start with the CheXNet init params cause why not
learning_rate = 0.0001 # decay by factor of 10 each time validation loss platueas after epoch
# lets also make it 10x smaller, 1e-4 should work better. 

subset = False # this can either make it smaller or bigger depending on our needs. 
modes = ["pneumonia", "abnormal"]
num_epochs = 23

for mode in modes:
    # Adam optimizer, standard params. 
    train_loader, test_loader, pos_weight = get_dataloaders(projections_path, reports_path, image_dir, mode=mode, subset=subset) # after that I got a bunch of defaults, I'll change em when I get there.

    loss_function = nn.BCEWithLogitsLoss(pos_weight=pos_weight)  # trying a different loss function, we shall see.
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate) # 
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5)

    modes = ["abnormal", "pneumonia"]
    history = train_model(model, train_loader, test_loader, num_epochs, device, pos_weight, mode=mode)

This is the projections path  ../IU-xray/indiana_projections.csv
Pneumonia mode: 676 positives / 3818 total (17.7%)
Best new model! AUC 0.7663
Epoch 1/2 | Train Loss: 1.1741 | Val Loss: 0.9208 | Val AUC: 0.7663
Best new model! AUC 0.7717
Epoch 2/2 | Train Loss: 1.1546 | Val Loss: 0.9528 | Val AUC: 0.7717
This is the projections path  ../IU-xray/indiana_projections.csv
Abnormal mode: 2431 positives / 3818 total (63.7%)
Best new model! AUC 0.2995
Epoch 1/2 | Train Loss: 0.5420 | Val Loss: 0.5278 | Val AUC: 0.2995
Best new model! AUC 0.3601
Epoch 2/2 | Train Loss: 0.5287 | Val Loss: 0.5227 | Val AUC: 0.3601


In [71]:
df = build_master_df_for_pneumonia(projections_path, reports_path)
print(f"Total images: {len(df)}")
print(f"Unique filenames: {df['filename'].nunique()}")
print(f"Duplicate filenames: {len(df) - df['filename'].nunique()}")

This is the projections path  ../IU-xray/indiana_projections.csv
Pneumonia mode: 676 positives / 3818 total (17.7%)
Total images: 3818
Unique filenames: 3818
Duplicate filenames: 0


In [72]:
df = build_master_df_for_pneumonia(projections_path, reports_path)
print(f"DataFrame shape: {df.shape}")  # Should be (3818, 2)
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")  # Should be tiny

This is the projections path  ../IU-xray/indiana_projections.csv
Pneumonia mode: 676 positives / 3818 total (17.7%)
DataFrame shape: (3818, 2)
Memory usage: 306.0 KB


In [73]:
# Load best model
model.load_state_dict(torch.load("Best_densenet_pneumonia.pth"))

# Final evaluation
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.sigmoid(outputs.squeeze())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

final_auc = roc_auc_score(all_labels, all_preds)
print(f"Final Test AUC: {final_auc:.4f}")

Final Test AUC: 0.3191
